# 3. Bands, timers and error messages

`examples/timers/` has equations in two bands: **1 Hz** and **10 Hz**. Bands are explicit frequencies with no required relationship. The engine tick rate is the least common multiple of all band frequencies, so every period is a whole number of ticks.

In [1]:
import sys, pathlib
try:
    import statesim
except ImportError:                      # not pip-installed: use the repo's src/ folder
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
    import statesim
from statesim import load_model, Engine, ListTraceSink, FileTraceWriter, ModelError, EvaluationError
print("statesim imported")

statesim imported


In [2]:
from statesim.scheduler import build_schedule

model = load_model("../examples/timers/master.csv")
print("tick rate:", model.tick_hz, "Hz")
for item in build_schedule(model):
    print(f"{item.name:<12} band={item.band_hz} Hz  order={item.order}  period={item.period_ticks} ticks")

tick rate: 10 Hz
fast_2s      band=10 Hz  order=1  period=1 ticks
fast_dt_100  band=10 Hz  order=2  period=1 ticks
both_2s      band=10 Hz  order=3  period=1 ticks
slow_2s      band=1 Hz  order=1  period=10 ticks


## `t` and `dt`
`t` is absolute simulation time in ms (0 at the first run). `dt` is the ms since the item last ran (on its first run: its period). Both are available in any equation. A timer is just `t >= 2000`.

Both timers go true at the same simulation time (2000 ms), regardless of band.

In [3]:
eng = Engine(model)
print(f"{'tick':>4} {'t(ms)':>7}  slow_2s fast_2s fast_dt_100 both_2s")
for _ in range(25):
    eng.step()
    s = eng.snapshot()
    o = s["outputs"]
    print(f"{s['tick']:>4} {s['t']:>7.0f}  {str(o['slow_2s']):<7} {str(o['fast_2s']):<7} {str(o['fast_dt_100']):<11} {o['both_2s']}")

tick   t(ms)  slow_2s fast_2s fast_dt_100 both_2s
   1       0  False   False   True        False
   2     100  False   False   True        False
   3     200  False   False   True        False
   4     300  False   False   True        False
   5     400  False   False   True        False
   6     500  False   False   True        False
   7     600  False   False   True        False
   8     700  False   False   True        False
   9     800  False   False   True        False
  10     900  False   False   True        False
  11    1000  False   False   True        False
  12    1100  False   False   True        False
  13    1200  False   False   True        False
  14    1300  False   False   True        False
  15    1400  False   False   True        False
  16    1500  False   False   True        False
  17    1600  False   False   True        False
  18    1700  False   False   True        False
  19    1800  False   False   True        False
  20    1900  False   False   True    

Look at tick 21 (t = 2000 ms): `slow_2s` and `fast_2s` both become true, but `both_2s` (10 Hz band, reads `slow_2s`) only becomes true one tick later. The 10 Hz band is the higher frequency, so on a tick where both bands are due it runs first and still sees `slow_2s` from the previous pass.

## Execution order
Within a band, items run in their manual `order`. A later item sees an earlier item's result **in the same pass** (see `both_2s`, order 3, reading `fast_2s`, order 1). Higher-frequency bands run first when several are due on the same tick.

## Load errors
Problems in the CSVs raise `ModelError` with a `file:line` source. These fixtures are in `tests/fixtures/`.

In [4]:
for name in ("bad_syntax", "bad_no_default", "bad_mixed_band"):
    try:
        load_model(f"../tests/fixtures/{name}/master.csv")
    except ModelError as e:
        print(f"{name:<16} -> {e}")

bad_syntax       -> equations.csv:2: invalid expression 'not go and': invalid syntax
bad_no_default   -> M.csv:1: no default_state specified
bad_mixed_band   -> M.csv:3: all rows of a state machine must share one band (20 != 10)


## Run-time errors
An undefined name is not checked at load time. It fails when evaluated, as an `EvaluationError` carrying the source, tick and item name.

In [5]:
import tempfile, pathlib
d = pathlib.Path(tempfile.mkdtemp())
(d / "master.csv").write_text("file_name,type,order\nequations.csv,equations,\n")
(d / "equations.csv").write_text(
    "equation,explanation,rationale,result_variable,band,order\n"
    "typo_name > 0,demo,demo,bad_eq,10,1\n")
eng = Engine(load_model(d / "master.csv"))
try:
    eng.step()
except EvaluationError as e:
    print("EvaluationError:", e)
    print("  source:", e.source, " tick:", e.tick, " item:", e.item)

EvaluationError: equations.csv:2: tick 1, bad_eq: NameError: name 'typo_name' is not defined
  source: equations.csv:2  tick: 1  item: bad_eq
